# Exam 2021, 8.00-13.00 for the course 1MS041 (Introduction to Data Science / Introduktion till dataanalys)

## Instructions:
1. Complete the problems by following instructions.
2. When done, submit this file with your solutions saved, following the instruction sheet.

This exam has 3 problems for a total of 40 points, to pass you need
20 points.

## Some general hints and information:
* Try to answer all questions even if you are uncertain.
* Comment your code, so that if you get the wrong answer I can understand how you thought
this can give you some points even though the code does not run.
* Follow the instruction sheet rigorously.
* This exam is partially autograded, but your code and your free text answers are manually graded anonymously.
* If there are any questions, please ask the exam guards, they will escalate it to me if necessary.
* I (Benny) will visit the exam room at around 10:30 to see if there are any questions.

## Tips for free text answers
* Be VERY clear with your reasoning, there should be zero ambiguity in what you are referring to.
* If you want to include math, you can write LaTeX in the Markdown cells, for instance `$f(x)=x^2$` will be rendered as $f(x)=x^2$ and `$$f(x) = x^2$$` will become an equation line, as follows
$$f(x) = x^2$$
Another example is `$$f_{Y \mid X}(y,x) = P(Y = y \mid X = x) = \exp(\alpha \cdot x + \beta)$$` which renders as
$$f_{Y \mid X}(y,x) = P(Y = y \mid X = x) = \exp(\alpha \cdot x + \beta)$$

## Finally some rules:
* You may not communicate with others during the exam, for example:
    * You cannot ask for help in Stack-Overflow or other such help forums during the Exam.
    * You may not communicate with AI's, for instance ChatGPT.
    * Your on-line and off-line activity is being monitored according to the examination rules.

## Good luck!

In [ ]:
# Insert your anonymous exam ID as a string in the variable below
examID="XXX"


---
## Exam vB, PROBLEM 1
Maximum Points = 8


## Probability warmup
Let's say we have an exam question which consists of $20$ yes/no questions. 
From past performance of similar students, a randomly chosen student will know the correct answer to $N \sim \text{binom}(20,11/20)$ questions. Furthermore, we assume that the student will guess the answer with equal probability to each question they don't know the answer to, i.e. given $N$ we define $Z \sim \text{binom}(20-N,1/2)$ as the number of correctly guessed answers. Define $Y = N + Z$, i.e., $Y$ represents the number of total correct answers.

We are interested in setting a deterministic threshold $T$, i.e., we would pass a student at threshold $T$ if $Y \geq T$. Here $T \in \{0,1,2,\ldots,20\}$.

1. [5p] For each threshold $T$, compute the probability that the student *knows* less than $10$ correct answers given that the student passed, i.e., $N < 10$. Put the answer in `problem11_probabilities` as a list.
2. [3p] What is the smallest value of $T$ such that if $Y \geq T$ then we are 90\% certain that $N \geq 10$?

In [5]:
from math import factorial
# Define the binomial coefficient
def binomial(n, k):
    return factorial(n) // (factorial(k) * factorial(n - k))
# Hint the PMF of N is p_N(k) where p_N is
p = 11/20
p_N = lambda n: binomial(20,n) * (1-p)**(20-n) * (p)**n   # P(N=n)

In [ ]:

# Part 1: 
# replace XXX to represent P(N < 10) for T = [0,1,2,...,20], i.e. your answer should be a list
# of length 21.
def p_Y_T_N(T, n):
    sum = 0
    for z in range(max(0, T-n), 20-n+1):
        sum += binomial(20-n, z) * (1/2)**(20-n)
    return sum

def prob_numerator(T):
    sum = 0
    for n in range(0, 10):
        sum += p_N(n) * p_Y_T_N(T, n)
    return sum

def prob_denominator(T):
    sum = 0
    for n in range(0, 21):
        sum += p_N(n) * p_Y_T_N(T, n)
    return sum

def prob_list():
    ls = []
    for t in range(0,21):
        ls.append(prob_numerator(t)/prob_denominator(t))
    return ls
problem11_probabilities = prob_list()

[0.24928935982841177, 0.24928935982832875, 0.24928935982261038, 0.2492893596354927, 0.24928935576839326, 0.24928929915834958, 0.2492886751893024, 0.24928330207958446, 0.24924628523366013, 0.24903902630299046, 0.2480856990043143, 0.24460820014975926, 0.2349439695781523, 0.21475641513175908, 0.18267139196620988, 0.1427252244707204, 0.10227042692681906, 0.06762809950564576, 0.041664724391227426, 0.024151134340423375, 0.013287462679601604]
1.0000000000000002
0.006109899266621893
0.24928935982841177
0.013287462679601604


In [14]:

# Part 2: Give an integer between 0 and 20 which is the answer to 2.
smallest_t = 0
for t in range(0, 21):
    if(prob_numerator(t)/prob_denominator(t) <= 0.1):
        smallest_t = t
        break
problem12_T = smallest_t
print(problem12_T)

17


---
## Exam vB, PROBLEM 2
Maximum Points = 8


## Random variable generation and transformation

The purpose of this problem is to show that you can implement your own sampler, this will be built in the following three steps:

1. [2p] Implement a Linear Congruential Generator where you tested out a good combination (a large $M$ with $a,b$ satisfying the Hull-Dobell (Thm 6.8)) of parameters. Follow the instructions in the code block.
2. [2p] Using a generator construct random numbers from the uniform $[0,1]$ distribution.
3. [4p] Using a uniform $[0,1]$ random generator, generate samples from 

$$p_0(x) = \frac{\pi}{2}|\sin(2\pi x)|, \quad x \in [0,1] \enspace .$$

Using the **Accept-Reject** sampler (**Algorithm 1** in TFDS notes) with sampling density given by the uniform $[0,1]$ distribution.

In [26]:

def problem2_LCG(size=None, seed = 0):
    """
    A linear congruential generator that generates pseudo random numbers according to size.
    
    Parameters
    -------------
    size : an integer denoting how many samples should be produced
    seed : the starting point of the LCG, i.e. u0 in the notes.
    
    Returns
    -------------
    out : a list of the pseudo random numbers
    """
    M = 512
    a = 9
    b = 1
    pseudo_numbers = []
    u = seed
    for i in range(0, size):
        u = (a*u+b) % M
        pseudo_numbers.append(u)
    return pseudo_numbers
problem2_LCG(100)

[1,
 10,
 91,
 308,
 213,
 382,
 367,
 232,
 41,
 370,
 259,
 284,
 509,
 486,
 279,
 464,
 81,
 218,
 427,
 260,
 293,
 78,
 191,
 184,
 121,
 66,
 83,
 236,
 77,
 182,
 103,
 416,
 161,
 426,
 251,
 212,
 373,
 286,
 15,
 136,
 201,
 274,
 419,
 188,
 157,
 390,
 439,
 368,
 241,
 122,
 75,
 164,
 453,
 494,
 351,
 88,
 281,
 482,
 243,
 140,
 237,
 86,
 263,
 320,
 321,
 330,
 411,
 116,
 21,
 190,
 175,
 40,
 361,
 178,
 67,
 92,
 317,
 294,
 87,
 272,
 401,
 26,
 235,
 68,
 101,
 398,
 511,
 504,
 441,
 386,
 403,
 44,
 397,
 502,
 423,
 224,
 481,
 234,
 59,
 20]

In [27]:

def problem2_uniform(generator=None, period = 1, size=None, seed=0):
    """
    Takes a generator and produces samples from the uniform [0,1] distribution according
    to size.
    
    Parameters
    -------------
    generator : a function of type generator(size,seed) and produces the same result as problem2_LCG, i.e. pseudo random numbers in the range {0,1,...,period-1}
    period : the period of the generator
    seed : the seed to be used in the generator provided
    size : an integer denoting how many samples should be produced
    
    Returns
    --------------
    out : a list of the uniform pseudo random numbers
    """
    
    uniform_pseudo_numbers = []
    for n in generator(size, seed = seed):
        uniform_pseudo_numbers.append(n/period)
    return uniform_pseudo_numbers

In [28]:

import math
def problem2_accept_reject(uniformGenerator=None, size=None,n_iterations=None, seed=0):
    """
    Takes a generator that produces uniform pseudo random [0,1] numbers 
    and produces samples from (pi/2)*abs(sin(x*2*pi)) using an Accept-Reject
    sampler with the uniform distribution as the proposal distribution
    
    Parameters
    -------------
    generator : a function of the type generator(size,seed) that produces uniform pseudo random
    numbers from [0,1]
    seed : the seed to be used in the generator provided
    size : an integer denoting how many samples should be produced
    
    Returns
    --------------
    out : a list of the pseudo random numbers with the specified distribution
    """
    if size is None:
        size = n_iterations
    
    samples = []
    while len(samples) < size:
        # Step 1: get 2 uniform numbers
        pair = uniformGenerator(size=2, seed=seed)
        x = pair[0]
        u = pair[1]

        # Step 2: acceptance condition
        if u <= abs(math.sin(2 * math.pi * x)):
            samples.append(x)
        
        # Step 3: advance seed so next iteration uses different numbers
        seed += 2

    return samples

---
#### Local Test for Exam vB, PROBLEM 2
Evaluate cell below to make sure your answer is valid.                             You **should not** modify anything in the cell below when evaluating it to do a local test of                             your solution.
You may need to include and evaluate code snippets from lecture notebooks in cells above to make the local test work correctly sometimes (see error messages for clues). This is meant to help you become efficient at recalling materials covered in lectures that relate to this problem. Such local tests will generally not be available in the exam.

In [29]:

# If you managed to solve all three parts you can test the following code to see if it runs
# you have to change the period to match your LCG though, this is marked as XXX.
# It is a very good idea to check these things using the histogram function in sagemath
# try with a larger number of samples, up to 10000 should run

print("LCG output: %s" % problem2_LCG(size=10, seed = 1))

period = 512

print("Uniform sampler %s" % problem2_uniform(generator=problem2_LCG, period = period, size=10, seed=1))

uniform_sampler = lambda size,seed: problem2_uniform(generator=problem2_LCG, period = period, size=size, seed=seed)

print("Accept-Reject sampler %s" % problem2_accept_reject(uniformGenerator = uniform_sampler,n_iterations=20,seed=1))

LCG output: [10, 91, 308, 213, 382, 367, 232, 41, 370, 259]
Uniform sampler [0.01953125, 0.177734375, 0.6015625, 0.416015625, 0.74609375, 0.716796875, 0.453125, 0.080078125, 0.72265625, 0.505859375]
Accept-Reject sampler [0.125, 0.16015625, 0.1953125, 0.23046875, 0.265625, 0.30078125, 0.3359375, 0.37109375, 0.58203125, 0.6171875, 0.6875, 0.72265625, 0.7578125, 0.79296875, 0.828125, 0.8984375, 0.93359375, 0.14453125, 0.1796875, 0.21484375]


In [ ]:

# If however you did not manage to implement either part 1 or part 2 but still want to check part 3, you can run the code below

def testUniformGenerator(size,seed):
    set_random_seed(seed)
    
    return [random() for s in range(size)]

print("Accept-Reject sampler %s" % problem2_accept_reject(uniformGenerator=testUniformGenerator, n_iterations=20, seed=1))

---
## Exam vB, PROBLEM 3
Maximum Points = 8


## Concentration of measure

As you recall, we said that concentration of measure was simply the phenomenon where we expect that the probability of a large deviation of some quantity becoming smaller as we observe more samples: [0.4 points per correct answer]

1. Which of the following will exponentially concentrate, i.e. for some $C_1,C_2,C_3,C_4 $ 
$$
    P(Z - \mathbb{E}[Z] \geq \epsilon) \leq C_1 e^{-C_2 n \epsilon^2} \wedge C_3 e^{-C_4 n (\epsilon+1)} \enspace .
$$

    1. The empirical mean of i.i.d. sub-Gaussian random variables?
    2. The empirical mean of i.i.d. sub-Exponential random variables?
    3. The empirical mean of i.i.d. random variables with finite variance?
    4. The empirical variance of i.i.d. random variables with finite variance?
    5. The empirical variance of i.i.d. sub-Gaussian random variables?
    6. The empirical variance of i.i.d. sub-Exponential random variables?
    7. The empirical third moment of i.i.d. sub-Gaussian random variables?
    8. The empirical fourth moment of i.i.d. sub-Gaussian random variables?
    9. The empirical mean of i.i.d. deterministic random variables?
    10. The empirical tenth moment of i.i.d. Bernoulli random variables?

2. Which of the above will concentrate in the weaker sense, that for some $C_1$
$$
    P(Z - \mathbb{E}[Z] \geq \epsilon) \leq \frac{C_1}{n \epsilon^2}?
$$

In [ ]:

# Answers to part 1, which of the alternatives exponentially concentrate, answer as a list
# i.e. [1,4,5] that is example 1, 4, and 5 concentrate
problem3_answer_1 = [1, 2, 5, 8, 9, 10]

In [ ]:

# Answers to part 2, which of the alternatives concentrate in the weaker sense, answer as a list
# i.e. [1,4,5] that is example 1, 4, and 5 concentrate
problem3_answer_2 = [1, 2, 3, 5, 6, 7, 8, 9, 10]

---
## Exam vB, PROBLEM 4
Maximum Points = 8


## SMS spam filtering [8p]

In the following problem we will explore SMS spam texts. The dataset is the `SMS Spam Collection Dataset` and we have provided for you a way to load the data. If you run the appropriate cell below, the result will be in the `spam_no_spam` variable. The result is a `list` of `tuples` with the first position in the tuple being the SMS text and the second being a flag `0 = not spam` and `1 = spam`.

1. [3p] Let $X$ be the random variable that represents each SMS text (an entry in the list), and let $Y$ represent whether text is spam or not i.e. $Y \in \{0,1\}$. Thus $\mathbb{P}(Y = 1)$ is the probability that we get a spam. The goal is to estimate:
$$
    \mathbb{P}(Y = 1 | \text{"free" or "prize" is in } X) \enspace .
$$
That is, the probability that the SMS is spam given that "free" or "prize" occurs in the SMS. 
Hint: it is good to remove the upper/lower case of words so that we can also find "Free" and "Prize"; this can be done with `text.lower()` if `text` a string.
2. [3p] Provide a "90\%" interval of confidence around the true probability. I.e. use the Hoeffding inequality to obtain for your estimate $\hat P$ of the above quantity. Find $l > 0$ such that the following holds:
$$
    \mathbb{P}(\hat P - l \leq \mathbb{E}[\hat P] \leq \hat P + l) \geq 0.9 \enspace .
$$
3. [2p] Repeat the two exercises above for "free" appearing twice in the SMS.

In [5]:

# Run this cell to get the SMS text data
#from exam_extras import load_sms
#spam_no_spam = load_sms()
spam_no_spam = [
    # SPAM with "free" or "prize"
    ("FREE entry in 2 a weekly comp to win FA Cup final tkts! Text FA to 87121", 1),
    ("You have WON a guaranteed FREE prize. Call 09061743386 to claim now", 1),
    ("Congratulations! You've won a FREE holiday. Call now to claim your prize", 1),
    ("FREE MESSAGE: You have been selected for a cash prize of 500 pounds", 1),
    ("Win a FREE car! Txt CAR to 80082. T&Cs apply. prize worth 20000", 1),
    ("URGENT! You have won a prize. Claim your FREE gift card now. Call 0800", 1),
    ("Get FREE ringtones! Send YES to 62735. Cost 3 pounds per week", 1),
    ("You are a WINNER! FREE iPod. Reply FREE to claim. Offer ends midnight", 1),
    ("FREE prize draw! You have been randomly selected. Text WIN to 84484", 1),
    ("Claim your FREE bonus now. Limited time offer. Prize expires tonight", 1),
    ("You've been chosen for a FREE luxury break. Call 0906 to claim prize", 1),
    ("Double your cash with our FREE bet offer! Prize guaranteed if you reply", 1),
    # SPAM without "free" or "prize"
    ("Urgent! Your mobile account needs verification. Call 0906 immediately", 1),
    ("You have 1 new voicemail. Call 0906 555 1111 to hear it. Cost 25p/min", 1),
    ("Congratulations! Call 09061234567 to claim your reward. T&Cs apply", 1),
    ("SIX chances to win CASH! From 100 to 20000 pounds txt CSH11 to 87575", 1),
    ("WINNER! As a valued customer you have been selected to receive a 900 prize", 1),
    ("Your loan is approved for 5000 pounds. Call 0800 to confirm today", 1),
    ("Hot singles in your area! Reply STOP to opt out. 3 pounds per msg", 1),
    ("Txt CALL to 85023 now for a secret surprise. T&Cs apply. 18+ only", 1),
    # NON-SPAM with "free" or "prize"
    ("Are you free tonight? We could grab dinner if you are", 0),
    ("The coffee in the break room is free today, someone left a whole box", 0),
    ("I got a free sample of that new shampoo, want to try it?", 0),
    ("Is the seminar free to attend or do we need to register?", 0),
    ("There's free parking behind the library on weekends", 0),
    ("The prize for the raffle was just a chocolate box haha", 0),
    ("She won first prize at the science fair, so proud of her", 0),
    ("Feel free to call me anytime this weekend", 0),
    ("We got free tickets to the match from work, you coming?", 0),
    # NON-SPAM without "free" or "prize"
    ("Hey, are you coming to the party on Saturday?", 0),
    ("Can you pick up some milk on your way home?", 0),
    ("The meeting has been moved to 3pm tomorrow", 0),
    ("I'll be there in about 20 minutes, traffic is bad", 0),
    ("Did you see the game last night? Incredible finish", 0),
    ("Happy birthday! Hope you have a wonderful day", 0),
    ("Could you send me the report when you get a chance?", 0),
    ("I'm running a bit late, start without me", 0),
    ("Thanks for dinner last night, it was lovely", 0),
    ("Can we reschedule our call to Thursday?", 0),
    ("The kids got home safely, just letting you know", 0),
    ("Reminder: dentist appointment tomorrow at 10am", 0),
    ("Do you have the address for the venue?", 0),
    ("Just landed, will call you when I get to the hotel", 0),
    ("Hope your interview went well today!", 0),
    ("We need to talk about the project timeline", 0),
    ("Lunch tomorrow works for me, see you at noon", 0),
    ("The package arrived this morning, thank you", 0),
    ("Can you babysit on Friday evening?", 0),
    ("Heading to the gym, back in an hour", 0),
    ("Good morning! Hope you have a great day", 0),
    ("The wifi password is on the fridge", 0),
    ("Sorry I missed your call, was in a meeting", 0),
    ("Are you watching the documentary tonight?", 0),
    ("I left my keys at your place, is that okay?", 0),
    ("Movie starts at 8, shall we meet at 7:30?", 0),
    ("Just checking in, how are you feeling?", 0),
    ("The weather looks nice this weekend finally", 0),
    ("Can you forward that email to me please?", 0),
    ("Dinner is ready whenever you get home", 0),
    # SPAM with "free" appearing TWICE
    ("FREE FREE ringtones! Text RING to 80082 now. Cost 1.50 per week", 1),
    ("Get FREE music and FREE wallpapers! Reply YES to 62735 to subscribe", 1),
    ("FREE offer: buy one get one FREE on all premium texts. Reply NOW", 1),
    ("Claim your FREE gift and FREE entry into prize draw. Text WIN to 84484", 1),
    # NON-SPAM with "free" appearing TWICE
    ("Feel free to bring a friend, the more the merrier, entry is free", 0),
    ("The app is free to download and free to use forever, no subscription", 0),
]

In [9]:

# fill in the estimate for part 1 here (should be a number between 0 and 1)
denominator = 0
numerator = 0
for t in spam_no_spam:
    if("free" in t[0].lower() or "prize" in t[0].lower()):
        denominator+=1
        if(t[1] == 1):
           numerator+=1
print(denominator, "messages contain 'free' or 'prize'")
print(numerator, "of those are spam")
problem4_hatP = numerator/denominator
print(problem4_hatP)

28 messages contain 'free' or 'prize'
17 of those are spam
0.6071428571428571


In [ ]:
# fill in the calculated l from part 2 here
import math
d = 0.1 # δ 
n = 28  # 28 messages contain 'free' or 'prize'
problem4_l = math.sqrt(math.log(2/d)/(2*n))
confidence_interval90 = (problem4_hatP - problem4_l, problem4_hatP + problem4_l )
print(confidence_interval90)

(0.3758525219132869, 0.8384331923724273)


In [12]:

# fill in the estimate for hatP for the double free question in part 3 here (should be a number between 0 and 1)
denominator2 = 0
numerator2 = 0
for t in spam_no_spam:
    if(t[0].lower().count("free") >= 2):
        denominator2+=1
        if(t[1] == 1):
           numerator2+=1
print(denominator2, "messages contain 'free' at lest two times")
print(numerator2, "of those are spam")
problem4_hatP2 = numerator2/denominator2
print(problem4_hatP2)

7 messages contain 'free' at lest two times
5 of those are spam
0.7142857142857143


In [14]:

# fill in the estimate for l for the double free question in part 3 here
import math
d = 0.1 # δ 
n = 7  # 28 messages contain 'free' or 'prize'
problem4_l2 = math.sqrt(math.log(2/d)/(2*n))
lower = max(0.0, problem4_hatP2 - problem4_l2)
upper = min(1.0, problem4_hatP2 + problem4_l2)
confidence_interval90 = (lower, upper)
print(confidence_interval90)

(0.2517050438265738, 1.0)


---
## Exam vB, PROBLEM 5
Maximum Points = 8


## Markovian travel

The dataset `Travel Dataset - Datathon 2019` is a simulated dataset designed to mimic real corporate travel systems -- focusing on flights and hotels. The file is at `data/flights.csv` in the same folder as `Exam.ipynb`, i.e. you can use the path `data/flights.csv` from the notebook to access the file.

1. [2p] In the first code-box 
    1. Load the csv from file `data/flights.csv`
    2. Fill in the value of the variables as specified by their names.
2. [2p] In the second code-box your goal is to estimate a Markov chain transition matrix for the travels of these users. For example, if we enumerate the cities according to alphabetical order, the first city `'Aracaju (SE)'` would correspond to $0$. Each row of the file corresponds to one flight, i.e. it has a starting city and an ending city. We model this as a stationary Markov chain, i.e. each user's travel trajectory is a realization of the Markov chain, $X_t$. Here, $X_t$ is the current city the user is at, at step $t$, and $X_{t+1}$ is the city the user travels to at the next time step. This means that to each row in the file there is a corresponding pair $(X_{t},X_{t+1})$. The stationarity assumption gives that for all $t$ there is a transition density $p$ such that $P(X_{t+1} = y | X_t = x) = p(x,y)$ (for all $x,y$). The transition matrix should be `n_cities` x `n_citites` in size.
3. [2p] Use the transition matrix to compute out the stationary distribution.
4. [2p] Given that we start in 'Aracaju (SE)' what is the probability that after 3 steps we will be back in 'Aracaju (SE)'?

In [9]:
import pandas as pd
df = pd.read_csv("/Users/secon/DocumentsLocal/Introduction_to_Data_Science/exams/data/flights.csv")
print(df.head())
combined_cities = pd.concat([df['from'], df['to']])
unique_cities = combined_cities.unique()
print(unique_cities)
number_of_cities = len(unique_cities)
print(number_of_cities)
number_of_userCodes = df['userCode'].nunique()
print(number_of_userCodes)
number_of_observations = len(df)
print(number_of_observations)

   travelCode  userCode                from                  to  flightType  \
0           0         0         Recife (PE)  Florianopolis (SC)  firstClass   
1           0         0  Florianopolis (SC)         Recife (PE)  firstClass   
2           1         0       Brasilia (DF)  Florianopolis (SC)  firstClass   
3           1         0  Florianopolis (SC)       Brasilia (DF)  firstClass   
4           2         0        Aracaju (SE)       Salvador (BH)  firstClass   

     price  time  distance       agency        date  
0  1434.38  1.76    676.53  FlyingDrops  09/26/2019  
1  1292.29  1.76    676.53  FlyingDrops  09/30/2019  
2  1487.52  1.66    637.56      CloudFy  10/03/2019  
3  1127.36  1.66    637.56      CloudFy  10/04/2019  
4  1684.05  2.16    830.86      CloudFy  10/10/2019  
['Recife (PE)' 'Florianopolis (SC)' 'Brasilia (DF)' 'Aracaju (SE)'
 'Salvador (BH)' 'Campo Grande (MS)' 'Sao Paulo (SP)' 'Natal (RN)'
 'Rio de Janeiro (RJ)']
9
1335
271888


In [10]:

# This is a very useful function that you can use for part 2. You have seen this before when parsing the
# pride and prejudice book.

def makeFreqDict(myDataList):
    '''Make a frequency mapping out of a list of data.

    Param myDataList, a list of data.
    Return a dictionary mapping each unique data value to its frequency count.'''

    freqDict = {} # start with an empty dictionary

    for res in myDataList:
        if res in freqDict: # the data value already exists as a key
                freqDict[res] = freqDict[res] + 1 # add 1 to the count using sage integers
        else: # the data value does not exist as a key value
            freqDict[res] = 1 # add a new key-value pair for this new data value, frequency 1

    return freqDict # return the dictionary created

In [19]:

cities = pd.concat([df['from'], df['to']])
unique_cities = sorted(set(cities)) # The unique cities
n_cities = len(unique_cities) # The number of unique citites

# Count the different transitions
transitions = list(zip(df['from'], df['to'])) # A list containing tuples ex: ('Aracaju (SE)','Rio de Janeiro (RJ)') of all transitions in the text
transition_counts = makeFreqDict(transitions) # A dictionary that counts the number of each transition 
# ex: ('Aracaju (SE)','Rio de Janeiro (RJ)'):4
indexToCity = {i: city for i, city in enumerate(unique_cities)} # A dictionary that maps the n-1 number to the n:th unique_city,
# ex: 0:'Aracaju (SE)'
cityToIndex = {city: i for i, city in enumerate(unique_cities)} # The inverse function of indexToWord, 
# ex: 'Aracaju (SE)':0

# Part 3, finding the maximum likelihood estimate of the transition matrix
import numpy as np
transition_matrix = np.zeros((n_cities, n_cities)) # a numpy array of size (n_cities,n_cities)

for i in range(n_cities):
    total = sum(transition_counts.get((indexToCity[i], indexToCity[k]), 0)
                for k in range(n_cities))
    if total > 0:
        for j in range(n_cities):
            transition_matrix[i, j] = (
                transition_counts.get((indexToCity[i], indexToCity[j]), 0) / total
            )
# Verify
print(transition_matrix.shape)
print(np.sum(transition_matrix, axis=1))
print(np.isnan(transition_matrix).any())
# The transition matrix should be ordered in such a way that
# p_{'Aracaju (SE)','Rio de Janeiro (RJ)'} = transition_matrix[cityToIndex['Aracaju (SE)'],cityToIndex['Rio de Janeiro (RJ)']]
# and represents the probability of travelling Aracaju (SE)->Rio de Janeiro (RJ)

# Make sure that the transition_matrix does not contain np.nan from division by zero for instance

(9, 9)
[1. 1. 1. 1. 1. 1. 1. 1. 1.]
False


In [20]:

# This should be a numpy array of length n_cities which sums to 1 and is all positive
eigvals, eigvecs = np.linalg.eig(transition_matrix.T)
idx = np.argmin(np.abs(eigvals - 1))
pi = np.real(eigvecs[:, idx])
stationary_distribution_problem5 = pi / np.sum(pi)
print(stationary_distribution_problem5)
print(np.sum(stationary_distribution_problem5))

[0.13690932 0.1132047  0.12780262 0.21081107 0.08752133 0.11210498
 0.06184532 0.06290826 0.0868924 ]
0.9999999999999998


In [21]:

# Compute the return probability for part 3 of problem 5
P3 = np.linalg.matrix_power(transition_matrix, 3)
a = cityToIndex['Aracaju (SE)']
return_probability_problem5 = P3[a, a]
print(return_probability_problem5)

0.13331717737273135


---
#### Local Test for Exam vB, PROBLEM 5
Evaluate cell below to make sure your answer is valid.                             You **should not** modify anything in the cell below when evaluating it to do a local test of                             your solution.
You may need to include and evaluate code snippets from lecture notebooks in cells above to make the local test work correctly sometimes (see error messages for clues). This is meant to help you become efficient at recalling materials covered in lectures that relate to this problem. Such local tests will generally not be available in the exam.

In [ ]:
# Once you have created all your functions, you can make a small test here to see
# what would be generated from your model.
import numpy as np

start = np.zeros(shape=(n_cities,1))
start[cityToIndex['Aracaju (SE)'],0] = 1

current_pos = start
for i in range(10): 
    random_word_index = np.random.choice(range(n_cities),p=current_pos.reshape(-1))
    current_pos = np.zeros_like(start)
    current_pos[random_word_index] = 1
    print(indexToCity[random_word_index],end='->')
    current_pos = (current_pos.T@transition_matrix).T

Aracaju (SE)->Campo Grande (MS)->Florianopolis (SC)->Sao Paulo (SP)->Brasilia (DF)->Rio de Janeiro (RJ)->Florianopolis (SC)->Recife (PE)->Sao Paulo (SP)->Florianopolis (SC)->

---
## Exam vB, PROBLEM 6
Maximum Points = 8


## Black box testing

In the following problem we will continue with our SMS spam / nospam data. This time we will try to approach the problem as a pattern recognition problem. For this particular problem I have provided you with everything -- data is prepared, split into train-test sets and a black-box model has been fitted on the training data and predicted on the test data. Your goal is to calculate test metrics and provide guarantees for each metric.

1. [2p] Compute precision for class 1 (see notes 8.3.2 for definition), then provide an interval using Hoeffding's inequality for a 95\% confidence.
2. [2p] Compute recall for class 1(see notes 8.3.2 for definition), then provide an interval using Hoeffding's inequality for a 95\% interval.
3. [2p] Compute accuracy (0-1 loss), then provide an interval using Hoeffding's inequality for a 95\% interval.
4. [2p] If we would have used a classifier with VC-dimension 3, would we have obtained a smaller interval for accuracy by using all data?

In [23]:

# The code below will load data, split the data into train and test and run a "black box" algorithm on it
# the result of the "black box" is stored in predictions_problem6, the true values will be stored in
# Y_test_problem6
"""
import exam_extras
from exam_extras import load_sms_problem6
X_problem6, Y_problem6 = load_sms_problem6()

X_train_problem6,X_test_problem6,Y_train_problem6,Y_test_problem6 = exam_extras.train_test_split(X_problem6,Y_problem6)
predictions_problem6 = exam_extras.knn_predictions(X_train_problem6,Y_train_problem6,X_test_problem6,k=4)
"""
import numpy as np
# exam_extras not available outside exam — stub with fake data
Y_test_problem6 = np.array([1,0,1,1,0,0,1,0,1,0])
predictions_problem6 = np.array([1,0,1,0,0,1,1,0,1,0])
TP = 4  # true positive 
FN = 1  # false negative
FP = 1  # false positive
TN = 4  # true negative

In [ ]:

# Compute the precision of predictions_problem6 with respect to Y_test_problem6
problem6_precision = TP / (TP + FP)

0.8


In [30]:

# Compute the interval length l of precision of predictions_problem6 with respect to Y_test_problem6, with the same definition of l as in problem 4
import math
d = 0.05 # δ 
n = TP + FP  # 28 messages contain 'free' or 'prize'
problem6_precision_l = math.sqrt(math.log(2/d)/(2*n))
lower = max(0.0, problem6_precision - problem6_precision_l)
upper = min(1.0, problem6_precision + problem6_precision_l)
confidence_interval90 = (lower, upper)
print(confidence_interval90)

(0.19263853809169484, 1.0)


In [31]:
# Repeat the same procedure but for recall
problem6_recall = TP / (TP + FN)

In [32]:

problem6_recall_l = math.sqrt(math.log(2/d) / (2*(TP + FN)))

In [33]:
# Repeat the same procedure but for accuracy or 0-1 loss
problem6_accuracy = (TP + TN) / (TP + FP + FN + TN)

In [34]:

problem6_accuracy_l = math.sqrt(math.log(2/d) / (2*(TP + FP + FN + TN)))

In [35]:

# Below you will calculate the interval parameter l for a classifier running on all data with a VC dimension of 3
# put the value in problem6_VC_l and answer problem_VC_smaller as True if the interval is smaller than the test-accuracy above
# if not answer False. Make sure you replace XXX with something even if you only answer one of them.
d_vc = 3
n_total = len(Y_test_problem6)   # in real exam: full dataset size (train+test)
delta = 0.05

problem6_VC_l = math.sqrt((8 / n_total) * (d_vc * math.log(2 * n_total / d_vc) + math.log(4 / delta)))
problem6_VC_smaller = problem6_VC_l < problem6_accuracy_l
print(problem6_VC_l, problem6_accuracy_l, problem6_VC_smaller)

2.8387865843464213 0.4294694083467376 False
